**CTE Limitations**

- CTEs can only be used in the current query scope
    - Cannot be referenced after the final SELECT statement
- Problematic if you need to reuse your virtual tables for different purposes
- Dependent virtual tables cannot be referenced individually making debugging more difficult

**Temp Tables Introduction**

- Temp tables are only limited by the current session
- Basic syntax uses SELECT INTO #
- Disadvantage is once a temp table is created, you cannot run the entire script again
    - Adding DROP TABLE alleviates issue
- When to use temp tables over CTEs
    - When you need to reference virtual tables in multiple outputs
    - When you need to join large datasets in virtual tables
    - When you need a script instead of a query

In [8]:
USE AW2019;

DROP TABLE IF EXISTS #Sales
SELECT
    OrderDate,
    OrderMonth  = DATEFROMPARTS (YEAR (OrderDate), MONTH (OrderDate), 1),
    TotalDue,
    OrderRank   = ROW_NUMBER() OVER (PARTITION BY DATEFROMPARTS (YEAR (OrderDate), MONTH (OrderDate), 1) ORDER BY TotalDue DESC)
INTO #Sales
FROM Sales.SalesOrderHeader


DROP TABLE IF EXISTS #AvgSalesMinusTop10
SELECT
    OrderMonth,
    TotalSales = SUM(TotalDue)
INTO #AvgSalesMinusTop10
FROM #Sales
WHERE OrderRank > 10
GROUP BY OrderMonth


DROP TABLE IF EXISTS #Purchases
SELECT 
    OrderDate,
    OrderMonth  = DATEFROMPARTS (YEAR (OrderDate), MONTH (OrderDate), 1),
    TotalDue,
    OrderRank   = ROW_NUMBER() OVER (PARTITION BY DATEFROMPARTS (YEAR (OrderDate), MONTH (OrderDate), 1) ORDER BY TotalDue DESC)
INTO #Purchases
FROM .Purchasing.PurchaseOrderHeader


DROP TABLE IF EXISTS #AvgPurchasesMinusTop10
SELECT
    OrderMonth,
    TotalPurchases = SUM(TotalDue)
INTO #AvgPurchasesMinusTop10
FROM #Purchases
WHERE OrderRank > 10
GROUP BY OrderMonth


SELECT
    A.OrderMonth,
    A.TotalSales,
    B.TotalPurchases
FROM #AvgSalesMinusTop10 A
	JOIN #AvgPurchasesMinusTop10 B
		ON A.OrderMonth = B.OrderMonth
ORDER BY A.OrderMonth

(31465 rows affected)

(38 rows affected)

(4012 rows affected)

(29 rows affected)

(27 rows affected)

Total execution time: 00:00:00.142

OrderMonth,TotalSales,TotalPurchases
2011-12-01,1019635.6747,7254.3006
2012-01-01,3622013.9215,220767.0679
2012-02-01,1141791.6116,7610.834
2012-03-01,2441839.1531,218226.7469
2012-04-01,1341386.2938,2496.2083
2012-05-01,2259194.0397,5744.3167
2012-06-01,3527254.7224,107628.3985
2012-07-01,2560587.9382,234.7418
2012-08-01,1534278.5579,14737.7394
2012-09-01,2903851.4579,3485.3792
